# iTantra model packs

Builds `.itpack` files for the app. Run cells top to bottom.

Before starting: upload `tools.zip` (the repo's `tools/` folder zipped) using the Files panel on the left.

Output lands in `/content/dist` and is copied to Google Drive `MyDrive/itantra_dist`.

In [ ]:
# 1. Unpack tools and mount Drive (Drive keeps outputs if the session dies)
!unzip -oq /content/tools.zip -d /content/
!ls /content/tools
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/itantra_dist /content/dist /content/work

In [ ]:
# 2. Install AI4Bharat NeMo fork (10-15 min). Restart the runtime if Colab asks.
!apt-get -qq install -y libsndfile1 ffmpeg
!git clone -q https://github.com/AI4Bharat/NeMo.git /content/NeMo
%cd /content/NeMo
!git checkout -q nemo-v2 && bash reinstall.sh
%cd /content
!pip -q install onnx onnxruntime sherpa-onnx soundfile jiwer huggingface_hub

In [ ]:
# 3. Smoke test: NeMo imports
import nemo.collections.asr as nemo_asr, onnxruntime, sherpa_onnx
print('nemo ok, sherpa-onnx', sherpa_onnx.__version__)

In [ ]:
# 4. STT packs. Start with hi + en; add the rest once those work on the phone.
LANGS = ['hi', 'en']  # later: 'bn','mr','gu','ta','te','kn','ml','or'
for lang in LANGS:
    !python /content/tools/stt/export_indicconformer.py --lang {lang} --out /content/work/stt/{lang}
    !python /content/tools/packs/build_pack.py stt --lang {lang} --dir /content/work/stt/{lang} --out /content/dist
    !cp /content/dist/{lang}-stt.itpack /content/drive/MyDrive/itantra_dist/

In [ ]:
# 5. Verify STT on a real recording. Upload e.g. hi_test.wav (16 kHz mono) first.
!python /content/tools/stt/verify_stt.py --dir /content/work/stt/hi --wav /content/hi_test.wav

In [ ]:
# 6. MMS voices (no login needed) for hi, gu, or, en
!python /content/tools/tts/fetch_mms.py --out /content/work/tts
for lang in ['hi', 'gu', 'or', 'en']:
    !python /content/tools/packs/build_pack.py tts --lang {lang} --dir /content/work/tts/_mms_repo/{lang} --out /content/dist
    !python /content/tools/tts/verify_tts.py --dir /content/work/tts/_mms_repo/{lang} --text "test" --wav /content/{lang}_tts.wav
    !cp /content/dist/{lang}-tts.itpack /content/drive/MyDrive/itantra_dist/

In [ ]:
# 7. rasa13 voice for bn mr ta te kn ml. Needs a HF token (read) and accepted terms on the model page.
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# 8. Export rasa13 and test Tamil. Listen to /content/ta_rasa.wav.
#    If this cell errors, copy the full error to Claude, and meanwhile run cell 9 (MMS fallback).
import json
!python /content/tools/tts/export_rasa13.py --out /content/work/tts/rasa13
sid = json.load(open('/content/work/tts/rasa13/speakers.json'))['ta']
!python /content/tools/tts/verify_tts.py --dir /content/work/tts/rasa13 --text "வணக்கம், இது ஒரு சோதனை செய்தி" --sid {sid} --wav /content/ta_rasa.wav
!python /content/tools/packs/build_pack.py tts --lang bn mr ta te kn ml --dir /content/work/tts/rasa13 --name rasa13-tts --out /content/dist
!cp /content/dist/rasa13-tts.itpack /content/drive/MyDrive/itantra_dist/

In [ ]:
# 9. FALLBACK only if cell 8 failed: MMS voices for the other six languages
for lang in ['bn', 'mr', 'ta', 'te', 'kn', 'ml']:
    !python /content/tools/packs/build_pack.py tts --lang {lang} --dir /content/work/tts/_mms_repo/{lang} --out /content/dist
    !cp /content/dist/{lang}-tts.itpack /content/drive/MyDrive/itantra_dist/

In [ ]:
# 10. Summary
!ls -lh /content/drive/MyDrive/itantra_dist